# P4 Redox Baseline

Independent RX-392 and Batt-P30K redox probe summary. This notebook reads the frozen artifacts and checks the unit and gate conventions.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score

root = Path.cwd()
merged = pd.read_csv(root / 'data' / 'processed' / 'redox_merged.csv')
summary = json.loads((root / 'probes' / 'p4_redox_summary.json').read_text(encoding='utf-8'))
print(merged['source'].value_counts().to_dict())

{'Batt-P30K': 29519, 'RX-392': 392}


In [2]:
rx = merged[merged['source'] == 'RX-392'].copy()
assert len(rx) == 392
assert {'oxidation_free_energy', 'reduction_free_energy'}.issubset(rx.columns)
print('RX rows:', len(rx))
print(rx[['IP', 'EA', 'oxidation_free_energy', 'reduction_free_energy']].describe().loc[['min', 'max']].to_string())

RX rows: 392
            IP        EA  oxidation_free_energy  reduction_free_energy
min   2.912773 -1.317925               2.516125              -7.145389
max  11.221582  7.145386              11.060807               0.266764


In [3]:
metrics = pd.DataFrame({
    target: {
        model: values['mae'] for model, values in summary['metrics'][target].items()
    }
    for target in ('oxidation_free_energy', 'reduction_free_energy')
})
print(metrics.to_string())
print('gate:', summary['gate'])

                 oxidation_free_energy  reduction_free_energy
linear                        0.290518               0.415337
scalar_gpr                    0.292735               0.409624
fingerprint_gpr               0.693011               0.669877
gate: {'threshold_mae': 0.15, 'unit': 'eV', 'criterion': 'best model per target has held-out MAE below threshold', 'best_model_mae': {'oxidation_free_energy': 0.2905180517963865, 'reduction_free_energy': 0.4096241620366996}, 'passed': False}


In [4]:
for feature, target in [('IP', 'oxidation_free_energy'), ('EA', 'reduction_free_energy')]:
    x = rx[feature].to_numpy(float)
    y = rx[target].to_numpy(float)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    print(feature, 'slope=', slope, 'intercept=', intercept, 'R2=', r2_score(y, pred), 'MAE=', mean_absolute_error(y, pred))

IP slope= 0.9539554967308024 intercept= -0.20899104863209703 R2= 0.9220066819683471 MAE= 0.27215269821808613
EA slope= -0.8298726539509633 intercept= -1.1053938860696937 R2= 0.7882694176484796 MAE= 0.41845808724655176


In [5]:
assert summary['sources']['ecw_308']['status'] == 'unavailable'
assert summary['sources']['ecw_308']['rows'] == 0
assert summary['units']['oxidation_free_energy'] == 'eV'
assert summary['units']['reduction_free_energy'] == 'eV'
assert summary['gate']['passed'] is False
print('308 status:', summary['sources']['ecw_308']['status'])
print('target unit: eV; gate threshold:', summary['gate']['threshold_mae'], 'eV')
print('gate passed:', summary['gate']['passed'])

308 status: unavailable
target unit: eV; gate threshold: 0.15 eV
gate passed: False
